# Katip Mask Mini v2 · a pretrained Turkish model, shrunk to browser size, taught by Base

**Why v2.** Mini v1 was trained from scratch and reached only ~55% of names on real Wikipedia text (Nano ~51%, Base ~90%).
~15k documents are not enough to learn Turkish from nothing. v2 starts from **DistilBERTurk** (dbmdz, MIT), which has
read general Turkish, then shrinks it and lets Katip Mask Base teach it:

1. **vocabulary pruning:** keep only the word pieces our text uses (plus every single character, so no word becomes unknown);
2. **layer dropping:** keep 2-3 of its 6 layers;
3. **distillation** from Base, on the same text as Mini v1.

Three candidates (~24-31 MB); the smallest within 0.5 point of the best *dev* score wins; test sets are scored once.

| | |
|---|---|
| Time | ~5 min smoke test, then ~1-1.5 h on a free T4 |
| Output | `katip-mask-mini-v2.zip` |
| Goal | Wikipedia names clearly above Mini v1's 55% (DistilBERTurk at full size, 68 MB, reached 73%) |

**How to run:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all.

In [ ]:
# 1 · Check the GPU, get the code. Run this cell first. (Takes ~1 min.)
import os, subprocess, sys
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE. In Colab: Runtime > Change runtime type > T4 GPU. In Kaggle: Settings > Accelerator > GPU T4.")
if not os.path.exists("katip"):
    subprocess.run("git clone -q --depth 1 -b legal-study https://github.com/fikrifirat/katip", shell=True, check=True)
os.chdir("katip")
subprocess.run(f"{sys.executable} -m pip install -q regex tokenizers onnx onnxruntime faker huggingface_hub 'transformers>=4.44' accelerate", shell=True, check=True)
print(subprocess.run("git log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# Optional · Hugging Face token, only needed to upload at the end.
# Colab: the key icon on the left > add a secret named HF_TOKEN (write access) and allow this notebook.
# Kaggle: Add-ons > Secrets > HF_TOKEN. Nothing is hard-coded here.
HF_TOKEN = None
try:
    from google.colab import userdata; HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
print("HF token:", "found" if HF_TOKEN else "none (fine: you can download the zip instead)")

## 2 · Smoke test (5 min)

In [ ]:
!cd lab && MODE=smoke python mini/train_mini_v2.py 2>&1 | grep -v Warning | tail -5

## 3 · Full run (~1-1.5 h). Keep the tab open.

In [ ]:
!cd lab && MODE=full python mini/train_mini_v2.py 2>&1 | grep -v Warning | tail -25

In [ ]:
import json, pandas as pd
r = json.load(open("lab/mini/runs/katip-mask-mini-v2_results.json"))
display(pd.DataFrame(r["candidates"]).drop(columns=["dir"]))
w = r["winner"]; t = r["test"]["wiki test"]
print(f"winner {w['name']}: {w['int8_mb']} MB, trust {w['trust']}")
print(f"Wikipedia names (test): recall {t['recall']:.1%}  precision {t['precision']:.1%}   (Mini v1 54.7%, Nano ~51%, Base ~90%)")

In [ ]:
# Save the result: a zip you can download (left panel > Files) and, if you set HF_TOKEN, a PRIVATE upload to Hugging Face.
# Make it public later on the model page (Settings > Visibility) after checking the scores.
import shutil, os
z = shutil.make_archive("katip-mask-mini-v2", "zip", "lab/mini/runs")
print("zip:", z, round(os.path.getsize(z) / 1e6, 1), "MB")
try:
    from google.colab import files; files.download(z)
except Exception:
    pass
UPLOAD = False   # set True to upload
if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo("f1k/katip-mask-mini", private=True, exist_ok=True)
    api.upload_folder(folder_path="lab/mini/runs", repo_id="f1k/katip-mask-mini", commit_message="trained in a notebook (katip-mask-mini-v2)")
    print("uploaded to https://huggingface.co/f1k/katip-mask-mini (private)")

**Next:** the zip goes to Downloads; say so in the Katip thread and it gets scored on KVKK-Bench.